# Solar Filament Segmentation Challenge 2026 — CUDA Panoptic Model

This notebook trains an instance-aware model on the supplied MAGFiLO annotations. Each annotator record remains an independent training/evaluation sample, while all records for a physical JPEG stay in the same split.

Every image and target remains native **2048×2048 grayscale**; there is no resizing or cropping. The semantic, instance-center, and pixel-to-center offset heads feed a panoptic U-Net. The model and batch width adapt to available CUDA memory, with `DataParallel` using one full image per GPU where multiple suitable GPUs are visible. Full mode uses all annotation records, a grouped 80/20 filename split, mixed precision, grayscale intensity augmentation, and early stopping. Validation predicts each physical JPEG once, scores every annotator record, selects thresholds on validation, and reports both record-level and physical-image metrics. Test inference remains disabled until the trained model and validation results are reviewed.


## 1. GPU setup and experiment profile

The saved profile is `full`: all 1,154 annotation records from 707 physical JPEGs are split by physical filename. CUDA device count and VRAM are discovered at runtime. The model uses base width 8 below 15 GiB per GPU, 16 from 15–22 GiB, and 20 above 22 GiB. When multiple GPUs with sufficient memory are available, `DataParallel` assigns one complete image to each device. The local 8 GiB RTX 3070 Ti uses width 8; Kaggle accelerator capacity is detected when its kernel starts.

Kaggle input discovery searches mounted datasets for the MAGFiLO annotation file. Full-frame images remain 2048×2048. CUDA mixed precision is enabled; if an accelerator still runs out of memory, reduce `BASE_CHANNELS` while retaining the full image.


In [2]:
from collections import defaultdict
import json
import random
import sys
import time
from pathlib import Path

local_site_packages = Path.cwd() / ".venv" / "Lib" / "site-packages"
if local_site_packages.exists():
    sys.path.insert(0, str(local_site_packages))
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from pycocotools import mask as coco_mask

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required; select a Kaggle GPU accelerator. CPU training is disabled.")
DEVICE = torch.device("cuda:0")
GPU_COUNT = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
GPU_MEMORY_GIB = [torch.cuda.get_device_properties(i).total_memory / 1024**3 for i in range(GPU_COUNT)]
MIN_GPU_MEMORY_GIB = min(GPU_MEMORY_GIB)
if MIN_GPU_MEMORY_GIB >= 22:
    BASE_CHANNELS = 20
elif MIN_GPU_MEMORY_GIB >= 15:
    BASE_CHANNELS = 16
else:
    BASE_CHANNELS = 8
# DataParallel receives one complete full-resolution image per GPU.
BATCH_SIZE = GPU_COUNT if GPU_COUNT > 1 and MIN_GPU_MEMORY_GIB >= 15 else 1
USE_DATA_PARALLEL = GPU_COUNT > 1 and BATCH_SIZE >= GPU_COUNT
NUM_WORKERS = 0
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DATA_ROOT = None
if DATA_ROOT is None:
    candidates = [Path.cwd() / "MAGFilo_1.0_Kaggle_2026", Path.cwd().parent / "MAGFilo_1.0_Kaggle_2026", Path("D:/filament-segmentation-2026/MAGFilo_1.0_Kaggle_2026")]
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates.extend(kaggle_root.glob("*/MAGFilo_1.0_Kaggle_2026"))
        candidates.extend(annotation_path.parent.parent for annotation_path in kaggle_root.rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    DATA_ROOT = next((candidate for candidate in candidates if (candidate / "train" / "train_images").exists() and (candidate / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json").exists()), candidates[0])
else:
    DATA_ROOT = Path(DATA_ROOT)
TRAIN_DIR = DATA_ROOT / "train"
TRAIN_IMAGES_DIR = TRAIN_DIR / "train_images"
TEST_IMAGES_DIR = DATA_ROOT / "test" / "test_images"
ANNOTATION_PATH = TRAIN_DIR / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
RUN_PROFILE = "full"
FULL_EPOCHS = 30
SMOKE_EPOCHS = 2
PATIENCE = 5
MIN_DELTA = 0.0005
SMOKE_SOURCE_IMAGE_LIMIT = 8
VALIDATION_FRACTION = 0.2
FOREGROUND_CONFIDENCE = 0.50
CENTER_CONFIDENCE = 0.25
MIN_INSTANCE_AREA = 128

print(f"PyTorch {torch.__version__} | CUDA runtime {torch.version.cuda}")
for gpu_id, (gpu_name, memory_gib) in enumerate(zip(GPU_NAMES, GPU_MEMORY_GIB)):
    print(f"CUDA device {gpu_id}: {gpu_name} | {memory_gib:.1f} GiB")
print(f"Config: base={BASE_CHANNELS}, batch={BATCH_SIZE}, DataParallel={USE_DATA_PARALLEL}")
print(f"Dataset root: {DATA_ROOT} | profile={RUN_PROFILE}")
for required_path in [TRAIN_IMAGES_DIR, TEST_IMAGES_DIR, ANNOTATION_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)
sns.set_theme(style="whitegrid", context="notebook")


PyTorch 2.11.0+cu128 | CUDA runtime 12.8
CUDA device 0: NVIDIA GeForce RTX 3070 Ti Laptop GPU | 8.0 GiB
Config: base=8, batch=1, DataParallel=False
Dataset root: D:\filament-segmentation-2026\MAGFilo_1.0_Kaggle_2026 | profile=smoke


## 2. Read annotations and audit source files

COCO image records are independent annotator examples. Records pointing to the same JPEG will always remain in the same split.


In [4]:
with ANNOTATION_PATH.open('r', encoding='utf-8') as file:
    coco_data = json.load(file)
image_records = coco_data['images']
annotations = coco_data['annotations']
categories = coco_data['categories']
image_record_by_id = {record['id']: record for record in image_records}
annotations_by_image_id = defaultdict(list)
for annotation in annotations:
    annotations_by_image_id[annotation['image_id']].append(annotation)
physical_files = sorted((path for path in TRAIN_IMAGES_DIR.glob('*') if path.is_file() and path.suffix.lower() in {'.jpeg', '.jpg'}))
physical_names = {path.name for path in physical_files}
json_names = {record['file_name'] for record in image_records}
records_per_file = pd.Series([record['file_name'] for record in image_records]).value_counts()
record_manifest = pd.DataFrame([{'image_id': record['id'], 'file_name': record['file_name'], 'width': int(record['width']), 'height': int(record['height']), 'date_captured': record.get('date_captured'), 'annotation_count': len(annotations_by_image_id[record['id']]), 'source_record_count': int(records_per_file[record['file_name']])} for record in image_records])
print('Annotation categories:')
display(pd.DataFrame(categories))
display(pd.DataFrame([{'metric': 'Physical train JPEGs', 'count': len(physical_files)}, {'metric': 'COCO image records / training samples', 'count': len(record_manifest)}, {'metric': 'Distinct physical filenames', 'count': record_manifest['file_name'].nunique()}, {'metric': 'Filament instance annotations', 'count': len(annotations)}, {'metric': 'Filenames with repeated annotation records', 'count': int((records_per_file > 1).sum())}, {'metric': 'Extra annotator records beyond one per filename', 'count': len(record_manifest) - record_manifest['file_name'].nunique()}, {'metric': 'Missing JPEGs referenced by JSON', 'count': len(json_names - physical_names)}, {'metric': 'Train JPEGs absent from JSON', 'count': len(physical_names - json_names)}]))
display(record_manifest.head())
actual_dimensions = {}
for path in physical_files:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise OSError(f'Could not decode {path}')
    actual_dimensions[path.name] = (image.shape[1], image.shape[0])
dimension_counts = pd.Series(list(actual_dimensions.values())).value_counts().rename_axis('(width, height)').rename('physical JPEGs').to_frame()
print('Decoded physical image dimensions:')
display(dimension_counts)
print("Every image record's declared dimensions agree with its JPEG:", all((actual_dimensions[row.file_name] == (row.width, row.height) for row in record_manifest.itertuples())))


Annotation categories:
  supercategory  id            name
0      filament   1            Left
1      filament   2           Right
2      filament   3  Unidentifiable
3      filament   4       Ambiguous
                                            metric  count
0                             Physical train JPEGs    707
1            COCO image records / training samples   1154
2                      Distinct physical filenames    707
3                    Filament instance annotations   8199
4       Filenames with repeated annotation records    296
5  Extra annotator records beyond one per filename    447
6                 Missing JPEGs referenced by JSON      0
7                     Train JPEGs absent from JSON      0
                  image_id  ... source_record_count
0  040301-20140609195854Bh  ...                   1
1  050103-20111116063134Lh  ...                   3
2  050102-20111116063134Lh  ...                   3
3  050101-20111116063134Lh  ...                   3
4  030101-20130

## 3. Native-resolution target rasterization

Polygon vertices map directly to the original JPEG coordinates. Semantic and instance masks, backbone lines, instance-center heatmaps, and pixel-to-center offsets are formed from the full-size annotations; no image or label crops are created. The instance-aware heads are optimized for the challenge's per-filament mask evaluation, while available category labels are an auxiliary semantic target.


In [6]:
category_ids = sorted({int(annotation["category_id"]) for annotation in annotations})
category_to_train_label = {
    category_id: index + 1 for index, category_id in enumerate(category_ids)
}
train_label_to_category = {
    train_label: category_id
    for category_id, train_label in category_to_train_label.items()
}
category_name_by_id = {
    int(category["id"]): category["name"] for category in categories
}
NUM_SEMANTIC_CLASSES = len(category_ids) + 1


def polygon_points(flat_coordinates, declared_width, declared_height, width, height):
    coordinates = np.asarray(flat_coordinates, dtype=np.float32)
    if coordinates.size < 6 or coordinates.size % 2:
        return None

    points = coordinates.reshape(-1, 2).copy()
    points[:, 0] *= width / declared_width
    points[:, 1] *= height / declared_height
    points[:, 0] = np.clip(points[:, 0], 0, width - 1)
    points[:, 1] = np.clip(points[:, 1], 0, height - 1)
    return np.rint(points).astype(np.int32)


def rasterize_record(image_id):
    record = image_record_by_id[image_id]
    filename = record["file_name"]
    width, height = actual_dimensions[filename]
    semantic = np.zeros((height, width), dtype=np.uint8)
    spine = np.zeros((height, width), dtype=np.uint8)
    instances = []

    for annotation in annotations_by_image_id[image_id]:
        category_id = int(annotation["category_id"])
        train_label = category_to_train_label[category_id]
        instance_mask = np.zeros((height, width), dtype=np.uint8)

        for polygon in annotation.get("segmentation", []):
            points = polygon_points(
                polygon,
                int(record["width"]),
                int(record["height"]),
                width,
                height,
            )
            if points is not None:
                cv2.fillPoly(instance_mask, [points.reshape(-1, 1, 2)], 1)

        unassigned_pixels = (instance_mask > 0) & (semantic == 0)
        semantic[unassigned_pixels] = train_label

        spine_points = polygon_points(
            annotation.get("spine") or [],
            int(record["width"]),
            int(record["height"]),
            width,
            height,
        )
        if spine_points is not None:
            cv2.polylines(
                spine,
                [spine_points.reshape(-1, 1, 2)],
                isClosed=False,
                color=1,
                thickness=2,
            )

        instances.append(
            {
                "category_id": category_id,
                "mask": instance_mask,
                "annotation_id": annotation["id"],
            }
        )

    return semantic, spine, instances


example_id = record_manifest.iloc[0]["image_id"]
example_record = image_record_by_id[example_id]
example_image = cv2.imread(
    str(TRAIN_IMAGES_DIR / example_record["file_name"]), cv2.IMREAD_GRAYSCALE
)
example_semantic, example_spine, example_instances = rasterize_record(example_id)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(example_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Native 2048×2048 image")
axes[1].imshow(example_semantic, cmap="tab10")
axes[1].set_title("Native semantic mask")
axes[2].imshow(example_image, cmap="gray", vmin=0, vmax=255)
axes[2].imshow(example_spine, cmap="autumn", alpha=0.85)
axes[2].set_title(f"Spines; {len(example_instances)} instances")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()
print(
    "Category mapping:",
    {category_name_by_id[key]: value for key, value in category_to_train_label.items()},
)
print("Image shape:", example_image.shape)


Category mapping: {'Left': 1, 'Right': 2, 'Unidentifiable': 3}
Image shape: (2048, 2048)


## 4. Select records and load full images lazily

Smoke mode temporarily selects a few physical filenames for quick development. Full mode includes all annotation records. Each sample reads one complete 2048×2048 JPEG and rasterizes its complete native-resolution target. The data loader processes one image at a time, limiting both host and GPU memory.


In [8]:
rng = np.random.default_rng(SEED)
if RUN_PROFILE == 'smoke':
    selected_count = min(SMOKE_SOURCE_IMAGE_LIMIT, record_manifest.file_name.nunique())
    selected_files = set(rng.choice(sorted(record_manifest.file_name.unique()), size=selected_count, replace=False))
    modeling_manifest = record_manifest[record_manifest.file_name.isin(selected_files)].reset_index(drop=True)
    epochs = SMOKE_EPOCHS
elif RUN_PROFILE == 'full':
    modeling_manifest = record_manifest.reset_index(drop=True)
    epochs = FULL_EPOCHS
else:
    raise ValueError('RUN_PROFILE must be smoke or full')
print(f'Selected {len(modeling_manifest)} records from {modeling_manifest.file_name.nunique()} physical images; native pixels retained.')
display(modeling_manifest[['annotation_count', 'source_record_count']].describe())


Selected 7 records from 4 physical images; native pixels retained.
       annotation_count  source_record_count
count          7.000000             7.000000
mean           8.571429             2.142857
std            3.994043             0.899735
min            5.000000             1.000000
25%            6.000000             1.500000
50%            7.000000             2.000000
75%           10.000000             3.000000
max           16.000000             3.000000


## 5. Split by physical filename

The 80/20 split groups repeated annotation records by filename, preventing identical pixels from appearing in both training and validation.


In [10]:
unique_files = np.array(sorted(modeling_manifest.file_name.unique()))
rng.shuffle(unique_files)
n_validation_files = max(1, int(round(len(unique_files) * VALIDATION_FRACTION)))
validation_files = set(unique_files[:n_validation_files])
train_files = set(unique_files[n_validation_files:])
train_indices = np.flatnonzero(~modeling_manifest.file_name.isin(validation_files).to_numpy())
validation_indices = np.flatnonzero(modeling_manifest.file_name.isin(validation_files).to_numpy())
assert train_files.isdisjoint(validation_files)
split_summary = pd.DataFrame([{'split': 'train', 'records': len(train_indices), 'physical_images': modeling_manifest.iloc[train_indices].file_name.nunique()}, {'split': 'validation', 'records': len(validation_indices), 'physical_images': modeling_manifest.iloc[validation_indices].file_name.nunique()}])
display(split_summary)
print('Train/validation filename overlap:', len(train_files & validation_files))


        split  records  physical_images
0       train        6                3
1  validation        1                1
Train/validation filename overlap: 0


## 6. Full-frame panoptic U-Net

The four-level encoder-decoder produces three full-resolution outputs: semantic logits, an instance-center heatmap, and per-pixel offsets that vote toward each filament center. The center and offset heads let postprocessing separate nearby instances without relying only on connected components. Base width is selected from per-GPU VRAM, CUDA mixed precision is used, and each GPU receives one complete image.


In [12]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
    )


class PanopticUNet(nn.Module):
    """Full-frame U-Net with semantic, instance-center and pixel-offset heads."""
    def __init__(self, num_classes, base_channels=8):
        super().__init__()
        b = base_channels
        self.encoder0 = conv_block(1, b)
        self.encoder1 = conv_block(b, 2*b)
        self.encoder2 = conv_block(2*b, 4*b)
        self.encoder3 = conv_block(4*b, 8*b)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = conv_block(8*b, 16*b)
        self.up3, self.decoder3 = nn.ConvTranspose2d(16*b, 8*b, 2, 2), conv_block(16*b, 8*b)
        self.up2, self.decoder2 = nn.ConvTranspose2d(8*b, 4*b, 2, 2), conv_block(8*b, 4*b)
        self.up1, self.decoder1 = nn.ConvTranspose2d(4*b, 2*b, 2, 2), conv_block(4*b, 2*b)
        self.up0, self.decoder0 = nn.ConvTranspose2d(2*b, b, 2, 2), conv_block(2*b, b)
        self.semantic_head = nn.Conv2d(b, num_classes, 1)
        self.center_head = nn.Conv2d(b, 1, 1)
        nn.init.constant_(self.center_head.bias, -2.19)
        self.offset_head = nn.Conv2d(b, 2, 1)

    def forward(self, image):
        e0 = self.encoder0(image)
        e1 = self.encoder1(self.pool(e0))
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        z = self.bottleneck(self.pool(e3))
        d3 = self.decoder3(torch.cat([self.up3(z), e3], 1))
        d2 = self.decoder2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.decoder1(torch.cat([self.up1(d2), e1], 1))
        d0 = self.decoder0(torch.cat([self.up0(d1), e0], 1))
        return self.semantic_head(d0), self.center_head(d0), self.offset_head(d0)


model = PanopticUNet(NUM_SEMANTIC_CLASSES, base_channels=BASE_CHANNELS).to(DEVICE)
if USE_DATA_PARALLEL:
    model = nn.DataParallel(model)
CLASS_WEIGHTS = torch.tensor([0.5] + [2.0] * len(category_ids), device=DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda")
print(f"Panoptic U-Net parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Model device:", next(model.parameters()).device)


Panoptic U-Net parameters: 486,463
Model device: cuda:0


## 7. Full-image loading, augmentation, and early stopping

Each sample is one complete original grayscale frame and its native-size targets. Horizontal and vertical flips update the offset directions as well as the image and masks. Mild gamma, contrast, and brightness changes model observatory variation without changing spatial detail. Mixed precision limits activation memory. Early stopping monitors the full-image validation loss and restores the best state; full mode allows 30 epochs with patience 5.


In [14]:
class FullImageDataset(Dataset):
    """One complete 2048x2048 grayscale JPEG and its full-size targets."""
    def __init__(self, record_indices, training=False):
        self.record_indices = np.asarray(record_indices, dtype=np.int64)
        self.training = training

    def __len__(self):
        return len(self.record_indices)

    @staticmethod
    def make_instance_targets(instances, height, width):
        center = np.zeros((height, width), np.float32)
        offset = np.zeros((2, height, width), np.float32)
        assigned = np.zeros((height, width), bool)
        scale = float(max(height, width))
        for instance in instances:
            mask = instance["mask"] > 0
            ys, xs = np.nonzero(mask)
            if len(xs) == 0:
                continue
            cx, cy = float(xs.mean()), float(ys.mean())
            ix, iy = int(round(cx)), int(round(cy))
            radius, sigma = 12, 4.0
            x0, x1 = max(0, ix-radius), min(width, ix+radius+1)
            y0, y1 = max(0, iy-radius), min(height, iy+radius+1)
            gy, gx = np.mgrid[y0:y1, x0:x1]
            gaussian = np.exp(-((gx-cx)**2 + (gy-cy)**2)/(2*sigma**2)).astype(np.float32)
            center[y0:y1, x0:x1] = np.maximum(center[y0:y1, x0:x1], gaussian)
            effective = mask & ~assigned
            if effective.any():
                ey, ex = np.nonzero(effective)
                offset[0, effective] = (cx-ex)/scale
                offset[1, effective] = (cy-ey)/scale
                assigned[effective] = True
        return center[None], offset

    def __getitem__(self, position):
        row = modeling_manifest.iloc[int(self.record_indices[position])]
        path = TRAIN_IMAGES_DIR / row["file_name"]
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if image is None or image.shape != (2048, 2048):
            raise ValueError(f"Expected 2048x2048 grayscale image: {path}; got {None if image is None else image.shape}")
        semantic, _, instances = rasterize_record(row["image_id"])
        center, offset = self.make_instance_targets(instances, *image.shape)
        image = image.astype(np.float32) / 255.0
        if self.training:
            if torch.rand(()) > 0.5:
                image, semantic, center = np.fliplr(image).copy(), np.fliplr(semantic).copy(), np.fliplr(center).copy()
                offset = np.flip(offset, axis=2).copy()
                offset[0] *= -1
            if torch.rand(()) > 0.5:
                image, semantic, center = np.flipud(image).copy(), np.flipud(semantic).copy(), np.flipud(center).copy()
                offset = np.flip(offset, axis=1).copy()
                offset[1] *= -1
            if torch.rand(()) < 0.65:
                gamma = float(torch.empty(()).uniform_(0.75, 1.35))
                contrast = float(torch.empty(()).uniform_(0.80, 1.20))
                brightness = float(torch.empty(()).uniform_(-0.08, 0.08))
                image = np.clip((image**gamma - 0.5)*contrast + 0.5 + brightness, 0, 1)
        return (
            torch.from_numpy(image[None].copy()).float(),
            torch.from_numpy(semantic.copy()).long(),
            torch.from_numpy(center.copy()).float(),
            torch.from_numpy(offset.copy()).float(),
        )


train_loader = DataLoader(FullImageDataset(train_indices, training=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
validation_loader = DataLoader(FullImageDataset(validation_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


def combined_loss(semantic_logits, center_logits, offset_prediction, semantic_target, center_target, offset_target):
    semantic_logits, center_logits = semantic_logits.float(), center_logits.float()
    offset_prediction, center_target, offset_target = offset_prediction.float(), center_target.float(), offset_target.float()
    cross_entropy = F.cross_entropy(semantic_logits, semantic_target, weight=CLASS_WEIGHTS)
    probability = semantic_logits.softmax(dim=1)
    one_hot = F.one_hot(semantic_target, NUM_SEMANTIC_CLASSES).permute(0, 3, 1, 2).float()
    intersection = (one_hot*probability).sum(dim=(2, 3))
    denominator = (one_hot+probability).sum(dim=(2, 3))
    present = one_hot.sum(dim=(2, 3)) > 0
    present[:, 0] = False
    dice = (2*intersection+1)/(denominator+1)
    dice_loss = 1-dice[present].mean() if present.any() else 0.0
    center_probability = center_logits.sigmoid().clamp(1e-6, 1-1e-6)
    positive = center_target >= 0.5
    negative_weight = (1-center_target).pow(4)
    center_positive = -torch.log(center_probability)*(1-center_probability).pow(2)
    center_negative = -torch.log(1-center_probability)*center_probability.pow(2)*negative_weight
    center_loss = center_positive[positive].sum()/positive.sum().clamp_min(1) + center_negative[~positive].mean()
    foreground = (semantic_target > 0)[:, None].expand_as(offset_prediction)
    offset_error = F.smooth_l1_loss(offset_prediction, offset_target, reduction="none")
    offset_loss = (offset_error*foreground).sum()/foreground.sum().clamp_min(1)
    return cross_entropy + dice_loss + center_loss + 5.0*offset_loss


def evaluate_loss(loader):
    model.eval()
    losses = []
    with torch.inference_mode():
        for images, semantic, center, offset in loader:
            images, semantic = images.to(DEVICE, non_blocking=True), semantic.to(DEVICE, non_blocking=True)
            center, offset = center.to(DEVICE, non_blocking=True), offset.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                outputs = model(images)
            losses.append(float(combined_loss(*outputs, semantic, center, offset).item()))
    return float(np.mean(losses))


print(f"Starting {RUN_PROFILE} training | GPUs={GPU_NAMES} | train records={len(train_indices)} | validation records={len(validation_indices)} | batch={BATCH_SIZE} | max epochs={epochs}")
print(f"Input: grayscale 2048x2048, no resize/crops | Panoptic U-Net base={BASE_CHANNELS} | AMP | patience={PATIENCE}")
history_rows, best_validation_loss, best_state, stale_epochs = [], float("inf"), None, 0
training_started = time.time()
for epoch in range(1, epochs+1):
    model.train()
    train_losses = []
    for batch_index, (images, semantic, center, offset) in enumerate(train_loader, start=1):
        images, semantic = images.to(DEVICE, non_blocking=True), semantic.to(DEVICE, non_blocking=True)
        center, offset = center.to(DEVICE, non_blocking=True), offset.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = model(images)
        loss = combined_loss(*outputs, semantic, center, offset)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(float(loss.detach().item()))
        if batch_index % 10 == 0 or batch_index == len(train_loader):
            print(f"  Epoch {epoch:02d}/{epochs} | batch {batch_index}/{len(train_loader)} | running loss={np.mean(train_losses):.4f}", flush=True)
    validation_loss = evaluate_loss(validation_loader)
    train_loss = float(np.mean(train_losses))
    history_rows.append({"epoch": epoch, "loss": train_loss, "val_loss": validation_loss})
    print(f"Epoch {epoch:02d}/{epochs} | train={train_loss:.4f} | validation={validation_loss:.4f} | allocated={torch.cuda.memory_allocated()/1024**3:.2f} GiB", flush=True)
    if validation_loss < best_validation_loss-MIN_DELTA:
        best_validation_loss, stale_epochs = validation_loss, 0
        module = model.module if isinstance(model, nn.DataParallel) else model
        best_state = {key: value.detach().cpu().clone() for key, value in module.state_dict().items()}
    else:
        stale_epochs += 1
        if RUN_PROFILE == "full" and stale_epochs >= PATIENCE:
            print(f"Early stopping after {PATIENCE} epochs without improvement.")
            break
if best_state is not None:
    (model.module if isinstance(model, nn.DataParallel) else model).load_state_dict(best_state)
model.eval()
history_frame = pd.DataFrame(history_rows)
if RUN_PROFILE == "full":
    CHECKPOINT_PATH = Path.cwd() / "solar_filament_panoptic_unet_best.pt"
    torch.save({"model_state_dict": (model.module if isinstance(model, nn.DataParallel) else model).state_dict(), "num_classes": NUM_SEMANTIC_CLASSES, "category_to_train_label": category_to_train_label, "image_size": (2048, 2048), "base_channels": BASE_CHANNELS, "num_gpus": GPU_COUNT, "architecture": "PanopticUNet_semantic_center_offset"}, CHECKPOINT_PATH)
    print(f"Saved full-run checkpoint: {CHECKPOINT_PATH}")
else:
    print("Smoke profile: checkpoint export skipped.")
print(f"Training finished in {(time.time()-training_started)/60:.2f} minutes; best validation loss={best_validation_loss:.4f}")


Starting smoke training | GPUs=['NVIDIA GeForce RTX 3070 Ti Laptop GPU'] | train records=6 | validation records=1 | batch=1 | max epochs=1
Input: grayscale 2048x2048, no resize/crops | Panoptic U-Net base=8 | AMP | patience=5
  Epoch 01/1 | batch 6/6 | running loss=3.9820
Epoch 01/1 | train=3.9820 | validation=4.4759 | allocated=0.16 GiB
Smoke profile: checkpoint export skipped.
Training finished in 0.86 minutes; best validation loss=4.4759


## 8. Training history

The plot shows full-image train and validation loss. Early stopping restores the model state with the lowest validation loss.


In [16]:
display(history_frame)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_frame.epoch, history_frame.loss, marker="o", label="train")
ax.plot(history_frame.epoch, history_frame.val_loss, marker="o", label="validation")
ax.set(xlabel="Epoch", ylabel="Full-frame multi-task loss", title="Native-resolution panoptic U-Net training")
ax.legend()
plt.tight_layout()
plt.show()


   epoch     loss  val_loss
0      1  3.98204  4.475924


## 9. Full-frame inference and instance decoding

Each original 2048×2048 image is one CUDA inference tensor. Semantic foreground probabilities define candidate filament pixels; local maxima of the center head propose instances, and offset votes assign pixels to centers. The category head is retained as an auxiliary output; mask matching and the submission RLE are category-agnostic, consistent with the competition's segmentation-mask objective.


In [18]:
def predict_probabilities_full(image):
    """Infer one complete 2048x2048 frame; never resize or crop it."""
    if image.shape != (2048, 2048):
        raise ValueError(f"Expected native 2048x2048 image; got {image.shape}")
    tensor = torch.from_numpy(image[None, None].astype(np.float32)/255.0).to(DEVICE)
    model.eval()
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        semantic_logits, center_logits, offset_prediction = model(tensor)
    semantic = semantic_logits.float().softmax(1)[0].permute(1, 2, 0).cpu().numpy()
    center = center_logits.float().sigmoid()[0, 0].cpu().numpy()
    offset = offset_prediction.float()[0].cpu().numpy()
    return semantic, center, offset


def split_instances(image, semantic, center, offset, foreground_threshold=FOREGROUND_CONFIDENCE, center_threshold=CENTER_CONFIDENCE):
    """Decode center votes into instance masks, then classify each instance."""
    height, width = image.shape
    classes = np.argmax(semantic, axis=-1)
    foreground = (1.0-semantic[..., 0]) >= foreground_threshold
    radius = 9
    maxima = center == cv2.dilate(center.astype(np.float32), np.ones((2*radius+1, 2*radius+1), np.uint8))
    ys, xs = np.nonzero(maxima & (center >= center_threshold))
    order = np.argsort(center[ys, xs])[::-1][:128]
    selected = []
    for idx in order:
        point = (int(xs[idx]), int(ys[idx]))
        if all((point[0]-x)**2+(point[1]-y)**2 >= 18**2 for x, y in selected):
            selected.append(point)
            if len(selected) >= 32:
                break
    if not selected:
        return []
    gy, gx = np.indices((height, width), dtype=np.float32)
    scale = float(max(height, width))
    vote_x, vote_y = gx+offset[0]*scale, gy+offset[1]*scale
    best_distance = np.full((height, width), np.inf, np.float32)
    assignment = np.full((height, width), -1, np.int32)
    for k, (cx, cy) in enumerate(selected):
        distance = (vote_x-cx)**2+(vote_y-cy)**2
        update = distance < best_distance
        best_distance[update], assignment[update] = distance[update], k
    instances = []
    for k in range(len(selected)):
        mask_full = (assignment == k) & foreground
        if int(mask_full.sum()) < MIN_INSTANCE_AREA:
            continue
        labels = classes[mask_full]
        labels = labels[labels > 0]
        if len(labels) == 0:
            continue
        category_id = train_label_to_category[int(np.bincount(labels).argmax())]
        my, mx = np.nonzero(mask_full)
        x0, x1, y0, y1 = int(mx.min()), int(mx.max())+1, int(my.min()), int(my.max())+1
        instances.append({"category_id": category_id, "mask": mask_full[y0:y1, x0:x1].astype(np.uint8), "bbox": (x0, y0, x1, y1)})
    return instances


def predict_record(image, foreground_threshold=FOREGROUND_CONFIDENCE, center_threshold=CENTER_CONFIDENCE):
    semantic, center, offset = predict_probabilities_full(image)
    instances = split_instances(image, semantic, center, offset, foreground_threshold, center_threshold)
    return semantic, center, offset, instances


## 10. Grouped validation metrics

Validation iterates over unique physical filenames and performs one model inference per JPEG, then compares that prediction with each corresponding annotator record. The local PQ matcher ignores auxiliary category labels and uses mask IoU strictly greater than 0.5. Results include record-micro and physical-image macro PQ, SQ/RQ, matched-instance Dice, foreground-pixel Dice/IoU, false positives, misses, and overlap-based fragmentation/merging counts. A small foreground/center threshold grid is tuned on this validation split, so the selected-threshold score is optimistic.


In [20]:
def instance_bbox(instance):
    if "bbox" in instance:
        return tuple(int(value) for value in instance["bbox"])
    ys, xs = np.nonzero(instance["mask"])
    return (int(xs.min()), int(ys.min()), int(xs.max())+1, int(ys.max())+1) if len(xs) else (0, 0, 0, 0)

def mask_iou_roi(a, b, ba, bb):
    x0, y0, x1, y1 = max(ba[0],bb[0]), max(ba[1],bb[1]), min(ba[2],bb[2]), min(ba[3],bb[3])
    if x1 <= x0 or y1 <= y0: return 0.0
    ar = a[y0-ba[1]:y1-ba[1], x0-ba[0]:x1-ba[0]]
    br = b[y0-bb[1]:y1-bb[1], x0-bb[0]:x1-bb[0]]
    inter = np.logical_and(ar > 0, br > 0).sum()
    union = int((a>0).sum()) + int((b>0).sum()) - inter
    return float(inter/union) if union else 0.0

def instance_metrics(gt, pred, threshold=0.5):
    gb, pb = [instance_bbox(x) for x in gt], [instance_bbox(x) for x in pred]
    pairs = []
    overlap_matrix = np.zeros((len(gt), len(pred)), dtype=np.float32)
    for gi, g in enumerate(gt):
        for pi, pred_i in enumerate(pred):
            iou = mask_iou_roi(g["mask"], pred_i["mask"], gb[gi], pb[pi])
            overlap_matrix[gi, pi] = iou
            if iou > threshold: pairs.append((iou, gi, pi))
    pairs.sort(reverse=True)
    used_g, used_p, matched = set(), set(), []
    for iou, gi, pi in pairs:
        if gi not in used_g and pi not in used_p:
            used_g.add(gi); used_p.add(pi); matched.append(iou)
    tp, fp, fn = len(matched), len(pred)-len(matched), len(gt)-len(matched)
    den = tp + 0.5*fp + 0.5*fn
    return {"PQ": sum(matched)/den if den else 1.0,
            "SQ": float(np.mean(matched)) if matched else 0.0,
            "RQ": tp/den if den else 1.0, "matched_instances": tp,
            "false_positive_instances": fp, "missed_instances": fn,
            "sum_matched_iou": sum(matched),
            "matched_mean_dice": float(np.mean([2*x/(1+x) for x in matched])) if matched else 0.0,
            "fragmented_ground_truth_instances": int(((overlap_matrix > 0).sum(axis=1) > 1).sum()),
            "overmerged_prediction_instances": int(((overlap_matrix > 0).sum(axis=0) > 1).sum())}

# Select thresholds on validation only. Use a separate untouched holdout for unbiased selection.
THRESHOLD_CANDIDATES = [(0.40,0.15), (0.50,0.25), (0.60,0.35)]
validation_groups = modeling_manifest.iloc[validation_indices].groupby("file_name", sort=True)
candidate_rows = {pair: [] for pair in THRESHOLD_CANDIDATES}
validation_previews = {}
print(f"Full-frame validation: {len(validation_groups)} distinct JPEGs and {len(validation_indices)} annotation records; infer each JPEG once.")
for file_n, (filename, records) in enumerate(validation_groups, 1):
    image = cv2.imread(str(TRAIN_IMAGES_DIR/filename), cv2.IMREAD_GRAYSCALE)
    if image is None or image.shape != (2048,2048): raise ValueError(f"Invalid validation JPEG: {filename}")
    semantic, center, offset = predict_probabilities_full(image)
    predictions_by_threshold = {
        pair: split_instances(image, semantic, center, offset, pair[0], pair[1])
        for pair in THRESHOLD_CANDIDATES
    }
    for row in records.itertuples(index=False):
        gt_semantic, _, gt = rasterize_record(row.image_id)
        gt_foreground = gt_semantic > 0
        for pair in THRESHOLD_CANDIDATES:
            fg_threshold, center_threshold = pair
            pred = predictions_by_threshold[pair]
            metrics = instance_metrics(gt, pred)
            pred_foreground = (1.0-semantic[...,0]) >= fg_threshold
            metrics.update({"image_id":row.image_id, "file_name":filename,
                "foreground_threshold":fg_threshold, "center_threshold":center_threshold,
                "pixel_intersection":int(np.logical_and(pred_foreground,gt_foreground).sum()),
                "pixel_predicted":int(pred_foreground.sum()), "pixel_ground_truth":int(gt_foreground.sum())})
            candidate_rows[pair].append(metrics)
            if pair == (0.50,0.25) and filename not in validation_previews and len(validation_previews)<4:
                validation_previews[filename] = {"image":image,"instances":pred,"ground_truth":gt,"image_id":row.image_id}
    print(f"Validation {file_n}/{len(validation_groups)}: {filename} | annotator records={len(records)} | one GPU inference", flush=True)

threshold_rows = []
for pair, rows in candidate_rows.items():
    frame = pd.DataFrame(rows)
    image_pq = frame.groupby("file_name").PQ.mean()
    threshold_rows.append({"foreground_threshold":pair[0], "center_threshold":pair[1],
        "macro_image_PQ":float(image_pq.mean()), "mean_annotation_record_PQ":float(frame.PQ.mean()),
        "mean_matched_instance_Dice":float(frame.matched_mean_dice.mean()),
        "physical_images":int(image_pq.size), "annotation_records":len(frame)})
threshold_summary = pd.DataFrame(threshold_rows).sort_values(["macro_image_PQ","mean_annotation_record_PQ"],ascending=False).reset_index(drop=True)
display(threshold_summary)
chosen = threshold_summary.iloc[0]
SELECTED_THRESHOLDS = (float(chosen.foreground_threshold),float(chosen.center_threshold))
FOREGROUND_CONFIDENCE, CENTER_CONFIDENCE = SELECTED_THRESHOLDS
if RUN_PROFILE == "full":
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
    checkpoint["foreground_threshold"], checkpoint["center_threshold"] = SELECTED_THRESHOLDS
    torch.save(checkpoint, CHECKPOINT_PATH)
    print(f"Updated checkpoint with selected thresholds: {CHECKPOINT_PATH}")
validation_metrics = pd.DataFrame(candidate_rows[SELECTED_THRESHOLDS])
tp, fp, fn = (int(validation_metrics[k].sum()) for k in ["matched_instances","false_positive_instances","missed_instances"])
sum_iou, den = float(validation_metrics.sum_matched_iou.sum()), tp+0.5*fp+0.5*fn
pix_inter, pix_pred, pix_gt = (int(validation_metrics[k].sum()) for k in ["pixel_intersection","pixel_predicted","pixel_ground_truth"])
pixel_dice = 2*pix_inter/(pix_pred+pix_gt) if pix_pred+pix_gt else 1.0
pixel_iou = pix_inter/(pix_pred+pix_gt-pix_inter) if pix_pred+pix_gt-pix_inter else 1.0
overall = {"PQ":sum_iou/den if den else 1.0,"SQ":sum_iou/tp if tp else 0.0,"RQ":tp/den if den else 1.0,
    "matched_instances":tp,"false_positive_instances":fp,"missed_instances":fn,
    "matched_mean_dice":float(validation_metrics.matched_mean_dice.mean()),
    "foreground_pixel_dice":pixel_dice,"foreground_pixel_iou":pixel_iou,
    "macro_physical_image_PQ":float(validation_metrics.groupby("file_name").PQ.mean().mean()),
    "validation_annotation_records":len(validation_metrics),"validation_physical_images":int(validation_metrics.file_name.nunique()),
    "unique_physical_images_inferred":int(validation_metrics.file_name.nunique())}
print(f"Selected thresholds: foreground={FOREGROUND_CONFIDENCE:.2f}, center={CENTER_CONFIDENCE:.2f}")
print("Validation metrics (record-micro and physical-image macro):")
display(pd.Series(overall,name="value").to_frame())
display(validation_metrics[["PQ","SQ","RQ","matched_mean_dice","matched_instances","false_positive_instances","missed_instances"]].describe())
print("The validation thresholds were tuned on this split, so selected-threshold validation scores are optimistic.")


Full-frame validation: 1 distinct JPEGs and 1 annotation records; infer each JPEG once.
Validation 1/1: 20161229133734Ch.jpeg | annotator records=1 | one GPU inference
   foreground_threshold  center_threshold  ...  physical_images  annotation_records
0                   0.4              0.15  ...                1                   1
1                   0.5              0.25  ...                1                   1
2                   0.6              0.35  ...                1                   1

[3 rows x 7 columns]
Selected thresholds: foreground=0.40, center=0.15
Validation metrics (record-micro and physical-image macro):
                                    value
PQ                               0.000000
SQ                               0.000000
RQ                               0.000000
matched_instances                0.000000
false_positive_instances         0.000000
missed_instances                 8.000000
matched_mean_dice                0.000000
foreground_pixel_dice       

## 11. Inspect native-resolution validation overlays and RLE

The overlays, ground-truth masks, and RLE round-trip remain at the original 2048×2048 resolution.


In [22]:
show_ids = list(validation_previews)
fig, axes = plt.subplots(len(show_ids), 3, figsize=(15, 5 * len(show_ids)))
if len(show_ids) == 1:
    axes = np.asarray([axes])
for r, iid in enumerate(show_ids):
    item = validation_previews[iid]
    gtc = np.zeros(item['image'].shape, np.uint16)
    prc = np.zeros_like(gtc)
    for j, x in enumerate(item['ground_truth'], 1):
        gtc[x['mask'] > 0] = j
    for j, x in enumerate(item['instances'], 1):
        bx, by, bx1, by1 = x['bbox']
        view = prc[by:by1, bx:bx1]
        view[x['mask'] > 0] = j
    axes[r, 0].imshow(item['image'], cmap='gray', vmin=0, vmax=255)
    axes[r, 0].set_title('Original image')
    axes[r, 1].imshow(gtc, cmap='nipy_spectral')
    axes[r, 1].set_title(f"Reference instances: {len(item['ground_truth'])}")
    axes[r, 2].imshow(prc, cmap='nipy_spectral')
    axes[r, 2].set_title(f"Predicted instances: {len(item['instances'])}")
    for ax in axes[r]:
        ax.axis('off')
plt.tight_layout()
plt.show()

def encode_instance_rle(mask):
    rle = coco_mask.encode(np.asfortranarray((mask > 0).astype(np.uint8)))
    return rle['counts'].decode('ascii') if isinstance(rle['counts'], bytes) else str(rle['counts'])
sample_mask = validation_previews[show_ids[0]]['ground_truth'][0]['mask']
rle = encode_instance_rle(sample_mask)
decoded = coco_mask.decode({'size': [2048, 2048], 'counts': rle.encode('ascii')})
assert np.array_equal(sample_mask, decoded)
print(f'COCO RLE round trip passed: 2048x2048, length {len(rle):,}.')


COCO RLE round trip passed: 2048x2048, length 283.


## 12. Optional native-resolution test prediction

No test masks or outside ground-truth metadata are used. After completing full training and reviewing grouped validation, set `WRITE_TEST_DRAFT=True` to predict the unlabeled test images and write one RLE row per predicted filament to the requested CSV format.


In [24]:
def encode_predicted_instance(instance):
    x, y, x1, y1 = instance['bbox']
    full = np.zeros((2048, 2048), np.uint8)
    full[y:y1, x:x1] = instance['mask']
    return encode_instance_rle(full)

def predict_test_to_csv(output_path='solar_filament_submission_draft.csv'):
    rows = []
    test_paths = sorted(TEST_IMAGES_DIR.glob('*.jpeg'))
    print(f"Running native-resolution inference on {len(test_paths)} test images.")
    for test_index, path in enumerate(test_paths, start=1):
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if image is None or image.shape != (2048, 2048):
            raise ValueError(f'Expected 2048x2048 test image: {path}')
        _, _, _, instances = predict_record(image)
        print(f"Test image {test_index}/{len(test_paths)}: {path.name} -> {len(instances)} instances", flush=True)
        for k, inst in enumerate(instances, 1):
            rows.append({'filament_id': f'{path.stem}_{k}', 'segmentation_rle': encode_predicted_instance(inst)})
    result = pd.DataFrame(rows, columns=['filament_id', 'segmentation_rle'])
    result.to_csv(output_path, index=False)
    return result
WRITE_TEST_DRAFT = False
if WRITE_TEST_DRAFT:
    draft_submission = predict_test_to_csv()
    display(draft_submission.head())
    print(f'Wrote {len(draft_submission)} instance rows at native resolution.')
else:
    print('Test prediction disabled; train and validate first, then set WRITE_TEST_DRAFT=True.')


Test prediction disabled; train and validate first, then set WRITE_TEST_DRAFT=True.


## 13. Final evaluation report

The final report summarizes the completed validation run. Smoke-mode values are only a quick pipeline check; the saved notebook defaults to all data and early-stopped training.


In [26]:
report = pd.DataFrame([
    ("Run profile", RUN_PROFILE), ("CUDA devices", "; ".join(GPU_NAMES)),
    ("GPU count / DataParallel", f"{GPU_COUNT} / {USE_DATA_PARALLEL}"),
    ("Image resolution", "2048x2048 grayscale; full frame; no resizing/cropping"),
    ("Batch size per replica", 1 if USE_DATA_PARALLEL else BATCH_SIZE),
    ("Panoptic U-Net base width", BASE_CHANNELS),
    ("Heads", "semantic classes + instance centers + pixel-to-center offsets"),
    ("Train physical images",len(train_files)),("Validation physical images",len(validation_files)),
    ("Train / validation annotation records",f"{len(train_indices)} / {len(validation_indices)}"),
    ("Train-validation filename overlap",len(train_files & validation_files)),
    ("Unique validation image inference count",overall["unique_physical_images_inferred"]),
    ("Epochs completed",len(history_frame)),("Early-stopping patience",PATIENCE),
    ("Best validation loss",best_validation_loss),("Selected thresholds",SELECTED_THRESHOLDS),
    ("Validation macro physical-image PQ",overall["macro_physical_image_PQ"]),
    ("Validation record-micro PQ",overall["PQ"]),("Validation SQ / RQ",f"{overall['SQ']:.4f} / {overall['RQ']:.4f}"),
    ("Matched instance Dice",overall["matched_mean_dice"]),
    ("Foreground pixel Dice / IoU",f"{overall['foreground_pixel_dice']:.4f} / {overall['foreground_pixel_iou']:.4f}"),
    ("Matched / false-positive / missed instances",f"{tp} / {fp} / {fn}"),
    ("GT instances with multiple overlaps",int(validation_metrics.fragmented_ground_truth_instances.sum())),
    ("Predicted instances with multiple overlaps",int(validation_metrics.overmerged_prediction_instances.sum())),
    ("Test CSV generated",WRITE_TEST_DRAFT)],columns=["Evaluation item","Result"])
print("\nFINAL VALIDATION EVALUATION REPORT")
print(report.to_string(index=False))
display(report)
print(f"\nFinal scores | macro image PQ={overall['macro_physical_image_PQ']:.4f} | record PQ={overall['PQ']:.4f} | instance Dice={overall['matched_mean_dice']:.4f}")
print("Review the overlays for fragmentation, merging, false positives, and missed filaments.")



FINAL VALIDATION EVALUATION REPORT
                            Evaluation item                                                        Result
                                Run profile                                                         smoke
                               CUDA devices                         NVIDIA GeForce RTX 3070 Ti Laptop GPU
                   GPU count / DataParallel                                                     1 / False
                           Image resolution         2048x2048 grayscale; full frame; no resizing/cropping
                     Batch size per replica                                                             1
                  Panoptic U-Net base width                                                             8
                                      Heads semantic classes + instance centers + pixel-to-center offsets
                      Train physical images                                                             3
          